# 経済データ分析Ⅱ　第3回
# 回帰分析の仮定と推定値の性質

前回は住宅データで重回帰を学び、二つの人口密度を同時に使うと係数と標準誤差が単回帰から大きく変わることを確かめた。今回は、OLS で求めた推定値が真の係数とどのような関係にあるのかを、回帰分析の標準的な仮定から考える。

実データでは真の係数が分からないので、推定値が正しいかを直接確かめられない。そこで今回は、所得と世帯人数から月額消費が決まる式を自分で決めて、人工データを作る。総務省の家計調査を参考に設計するので、数値の大きさは日本の勤労者世帯に近いが、実際の世帯の記録ではない。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中から始めて変数名のエラーが出たら、先頭のセルから実行し直してください。説明文の記入欄は、ダブルクリックすると編集できます。

ブラウザで使う場合は、このファイルを JupyterLite にアップロードし、**Python (Pyodide)** を選びます。初回のライブラリ読み込みには時間がかかることがあります。ローカルの Jupyter では Python カーネルを使います。各回は単独で実行でき、CSV ファイルの準備は不要です。

コードを書き換えたら必ず再実行してください。画面の出力は、再実行するまで更新されません。授業の終わりには Notebook を保存し、ブラウザで使った場合は `.ipynb` をダウンロードして手元にも残しましょう。

図中の英語は、`Income`＝所得、`Consumption`＝消費、`Household size`＝世帯人数、`Residual`＝残差、`Multiple regression`＝重回帰、`Simple regression`＝単回帰、`Estimated income coefficient`＝所得の係数の推定値です。金額の単位 `10,000 JPY / month` は「月額・万円」を表します。

## 1. 準備

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, MultipleLocator
import statsmodels.api as sm
from IPython.display import display

# 表示だけを丸める。計算には元の精度の数値を使う。
def format_table_number(value):
    text = f'{value:.3f}'
    # 表示上だけの負のゼロを避ける。データ自体は変更しない。
    return '0.000' if text == '-0.000' else text

pd.options.display.float_format = format_table_number
plt.rcParams.update({
    'figure.figsize': (8, 4.5), 'figure.dpi': 110,
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.2,
})

# 金額の軸は万円の整数、係数の軸は小数2桁で表示する。
def format_axes(ax, x_money=True, y_money=True):
    ax.xaxis.set_major_formatter(StrMethodFormatter('{x:.0f}' if x_money else '{x:.2f}'))
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.0f}' if y_money else '{x:.2f}'))

## 2. 家計調査を参考にした人工データ

200世帯について、世帯人数・月額所得・月額消費を順に作る。数値の大きさは、次の公表値を参考に決めた。

| 世帯人数 | 世帯の割合 | 平均所得（月額） | 平均消費（月額） |
|---|---|---|---|
| 1人 | 38.0% | 37.0万円 | 18.4万円 |
| 2人 | 28.1% | 54.1万円 | 29.9万円 |
| 3人 | 16.6% | 63.0万円 | 31.9万円 |
| 4人 | 11.9% | 72.2万円 | 34.5万円 |
| 5人 | 5.4% | 69.3万円 | 36.1万円 |

世帯の割合は総務省『国勢調査』（2020年、一般世帯。5人の行は5人以上の世帯の割合）、平均所得と平均消費は総務省『家計調査』（2024年平均、勤労者世帯の実収入と消費支出）による。

家計調査で所得の内訳を見ると、1人世帯と4人世帯の差の約9割は、世帯主の収入の差と配偶者の収入（共働き）である。子どもや祖父母など世帯主と配偶者以外の収入は月2万円以下である。5人世帯の平均所得は4人世帯より少し低い。

### 2.1. 乱数の準備

人工データは乱数（でたらめに選ばれた数）を使って作る。`np.random.default_rng(202610)` は乱数を作る仕組みを用意し、`rng` という名前を付ける。括弧の中の数（シード）を同じにしておくと、何度実行しても同じ乱数が出るので、全員が同じデータで学習できる。`n` は作る世帯の数である。

In [ ]:
rng = np.random.default_rng(202610)
n = 200

### 2.2. 世帯人数

`rng.choice(候補, size=個数, p=割合)` は、候補の中から、指定した割合に従って値を選ぶ。ここでは1〜5人の候補から、国勢調査の割合で200世帯分を選ぶ。`pd.Series(...).value_counts()` は、それぞれの値がいくつあるかを数える。

In [ ]:
sizes = np.array([1, 2, 3, 4, 5])
share = np.array([0.380, 0.281, 0.166, 0.119, 0.054])
household_size = rng.choice(sizes, size=n, p=share)
pd.Series(household_size).value_counts().sort_index()

1人世帯が最も多く、人数が多いほど世帯数は少ない。割合に従って選んでいるので、世帯数は表の割合に近いが、ちょうど一致はしない。

### 2.3. 所得

所得は次の2段階で作る。

- `mean_income` は、世帯人数ごとの平均所得（家計調査の値、万円）を並べたものである。`mean_income[household_size - 1]` で、各世帯の人数に対応する平均所得を取り出す。Python では先頭を0番目と数えるので、人数から1を引いている。
- 同じ人数の世帯でも所得は違うので、平均がほぼ1になる倍率 `multiplier` を掛けて、世帯ごとにばらつかせる。`rng.lognormal()` は対数正規分布から乱数を選ぶ。対数正規分布は正の値だけをとり、ときどき大きな値が出る分布で、所得のばらつきを表すのによく使われる。

In [ ]:
mean_income = np.array([37.0, 54.1, 63.0, 72.2, 69.3])
multiplier = rng.lognormal(mean=-0.06, sigma=0.35, size=n)
income = mean_income[household_size - 1] * multiplier
print(f'倍率の平均：{multiplier.mean():.3f}')
print(f'所得の最小：{income.min():.1f} 万円、最大：{income.max():.1f} 万円')

### 2.4. 消費と誤差

消費は、所得と世帯人数で決まる部分に、誤差 $u_i$ を足して作る。

$$消費_i=5+0.3\times 所得_i+2\times 世帯人数_i+u_i$$

所得が1万円多いと消費は平均0.3万円（3,000円）多く、世帯人数が1人多いと平均2万円多い。係数は、世帯人数ごとの平均消費が家計調査の値に近くなるように決めた。誤差は、式に入れていない要因（年齢、資産、好みなど）による消費の違いを表す。`rng.normal(0, 4, size=n)` で、平均0・標準偏差4万円の正規分布から、世帯ごとに独立に作る。

In [ ]:
true_error = rng.normal(0, 4, size=n)
consumption = 5 + 0.3 * income + 2 * household_size + true_error

df = pd.DataFrame({'income': income, 'household_size': household_size,
                   'consumption': consumption})
display(df.head())

`income`（月額所得、万円）、`household_size`（世帯人数、人）、`consumption`（月額消費、万円）の3列の表ができた。世帯人数ごとに平均を求め、家計調査の値と比べる。`groupby()` は、同じ世帯人数の行をまとめて計算する。

In [ ]:
summary = df.groupby('household_size').agg(
    世帯数=('income', 'size'),
    所得の平均=('income', 'mean'),
    消費の平均=('consumption', 'mean'),
)
summary['家計調査の所得'] = mean_income
summary['家計調査の消費'] = [18.4, 29.9, 31.9, 34.5, 36.1]
# この表だけ小数第1位まで表示する。
with pd.option_context('display.float_format', '{:.1f}'.format):
    display(summary)

所得の平均は、2人と3人の世帯で家計調査の値にほぼ一致し、1人と4人の世帯では5万円ほど違う。5人世帯は8世帯しかないので、平均が偶然に大きくずれやすい。消費の平均は1人と4人の世帯で家計調査に近く、2人世帯では4万円ほど低い。家計調査の2人世帯には年齢の高い夫婦が多く、所得に比べて消費が多いが、今回の単純な式ではそこまで表せない。

散布図で、所得と消費の関係を世帯人数ごとに色分けして確かめる。

In [ ]:
fig, ax = plt.subplots()
for size in sorted(df['household_size'].unique()):
    group = df[df['household_size'] == size]
    ax.scatter(group['income'], group['consumption'], s=25, alpha=0.65,
               label=f'{size} person(s)')
ax.set(xlabel='Income (10,000 JPY / month)', ylabel='Consumption (10,000 JPY / month)')
ax.legend(title='Household size', fontsize=9)
format_axes(ax)
plt.tight_layout()
plt.show()
print(f'所得と世帯人数の相関係数：{df[["income", "household_size"]].corr().iloc[0, 1]:.3f}')

所得が高い世帯ほど消費も多い。同じ所得でも、人数の多い世帯の点は上側に来やすい。所得と世帯人数の相関係数は約0.43で、人数の多い世帯ほど所得もやや高い。

以下では、消費を $y_i$、所得を $x_{1i}$、世帯人数を $x_{2i}$ と書く。$i$ は世帯の番号である。重回帰モデルは第2回と同じ形の

$$y_i=\beta_0+\beta_1x_{1i}+\beta_2x_{2i}+u_i$$

で、このデータの真の係数は $\beta_0=5$、$\beta_1=0.3$、$\beta_2=2$ である。第2回と同じ方法で重回帰を推定し、推定値を真の値と並べる。

In [ ]:
X_multiple = sm.add_constant(df[['income', 'household_size']])
multiple = sm.OLS(df['consumption'], X_multiple).fit()
display(pd.DataFrame({'推定値': multiple.params,
                      '真の値': [5, 0.3, 2],
                      '通常の標準誤差': multiple.bse}))

推定値は真の値に近いが、ぴったりとは一致しない。200世帯の誤差 $u_i$ に偶然のばらつきがあるためで、別の200世帯を調べれば推定値も変わる。推定値が平均的に真の値に一致するか、どのくらいばらつくかは、データがどのように生まれたかによる。そこで、データの生まれ方について仮定を置く。

## 3. 回帰分析の標準的な仮定

重回帰分析では、次の6つを標準的な仮定とする。それぞれ、何を求めているのかと、成り立たない例を添える。

- **仮定1（母数について線形）**：母集団で $y_i=\beta_0+\beta_1x_{1i}+\beta_2x_{2i}+u_i$ が成り立つ。消費の平均が、所得と世帯人数の一次式で表せるという意味である。係数について一次式であればよいので、所得の二乗や対数を説明変数に加えることはできる。所得が増えると消費の増え方が小さくなるのに直線だけを使う場合などは、この仮定が成り立たない。
- **仮定2（無作為抽出）**：各世帯は、同じ母集団から互いに独立に選ばれる。くじ引きのように偶然に選んだ世帯を調べるという意味である。高所得の世帯だけを集めたデータや、同じ世帯を何度も数えたデータでは成り立たない。
- **仮定3（完全な多重共線性がない）**：どの説明変数も、ほかの説明変数の足し算や定数倍では完全には表せない。月額所得と年額所得（月額の12倍）を同時に入れると成り立たず、二つの係数を一つに決められない。
- **仮定4（誤差の条件付き平均が0）**：所得や世帯人数がどの値でも、誤差 $u_i$ の平均は0である。式では $E[u_i\mid x_{1i},x_{2i}]=0$ と書く。誤差には式に入れていない要因（年齢、資産、好みなど）が含まれるので、この仮定は、それらの要因が所得や世帯人数と関係しないことを求めている。資産の多い世帯ほど所得が高く消費も多い場合は成り立たない。
- **仮定5（均一分散）**：所得や世帯人数がどの値でも、誤差の分散は同じ値 $\sigma^2$ である。所得の高い世帯ほど消費のばらつきが大きい場合は成り立たない。
- **仮定6（正規性）**：所得や世帯人数の値によらず、誤差は平均0・分散 $\sigma^2$ の正規分布に従う。

仮定を加えるごとに、OLS の推定値について言えることが増える。

- 仮定1〜4の下で、OLS 推定量は**不偏**である。不偏とは、標本を取り直して推定を何度も繰り返したとき、推定値の平均が真の係数に一致することをいう。1回ごとの推定値は真の値の上下にばらつく。
- 仮定1〜5の下で、推定値のばらつき（分散）は第5節の式で表され、statsmodels が表示する通常の標準誤差を使える。この条件の下では、OLS は線形で不偏な推定方法の中で最もばらつきが小さい（ガウス・マルコフの定理）。
- 仮定1〜6の下で、標本が小さくても $t$ 検定と信頼区間を正確に計算できる。仮定6が成り立たなくても、標本が大きければ近似として使える。

不偏性に必要なのは仮定1〜4で、仮定5と仮定6は使わない。中でも仮定4が重要で、これが成り立たないと、推定値が真の係数から系統的にずれることがある（第6節）。

今回の人工データは仮定1〜6をすべて満たすように作っている。実データで仮定4が成り立つかは、データの成り立ちや経済的な関係から判断する。次の節で見るように、残差からは確かめられない。

## 4. 誤差項と残差

- **誤差項** $u_i$：母集団の式からのずれ。真の係数が分からないため、実データでは計算できない。
- **残差** $e_i$：推定した式からのずれ。観測値と推定結果から計算できる。

今回は真の式を決めた人工データなので、両方を並べて比べられる。

In [ ]:
error_comparison = df.head(6).copy()
error_comparison['誤差項'] = true_error[:6]
error_comparison['残差'] = multiple.resid.iloc[:6].to_numpy()
display(error_comparison)
print(f'誤差項の平均：{format_table_number(true_error.mean())}')
print(f'残差の平均：{format_table_number(multiple.resid.mean())}')

誤差項と残差は近い値だが、少しずつ違う。平均0の分布から作った誤差でも、200個の平均はちょうど0にはならない。一方、残差の平均は0になる。

### 4.1. 残差が必ず満たす性質

OLS の残差は、次の3つの式を必ず満たす。

$$\sum_{i=1}^{n}e_i=0,\qquad \sum_{i=1}^{n}x_{1i}e_i=0,\qquad \sum_{i=1}^{n}x_{2i}e_i=0$$

1つ目は残差の合計が0であること、2つ目と3つ目は、残差が所得とも世帯人数とも相関しないことを表す。この3つの式は、データの生まれ方に関係なく OLS で係数を求めれば必ず成り立つ性質である。

なぜ成り立つのかを、所得の係数で考える。もし残差が所得と正の相関を持っていたら、所得の高い世帯ほど実際の消費が当てはめ値より上にあることになる。この場合、所得の係数を少し大きくすれば、当てはめ値が実際の消費に近づき、残差二乗和は小さくなる。OLS は残差二乗和が最も小さくなる係数を選ぶので、選ばれた係数では、そのような相関は残らない。第1回で残差二乗和を傾きで偏微分して0とおいた式が、2つ目の式にあたる。

所得の係数を推定値から少し動かし、残差二乗和が増えることを確かめる。ほかの係数は推定値のままにする。

In [ ]:
b = multiple.params
for change in [-0.02, 0, 0.02]:
    slope = b['income'] + change
    fitted = b['const'] + slope * df['income'] + b['household_size'] * df['household_size']
    ssr = ((df['consumption'] - fitted)**2).sum()
    print(f'所得の係数 {slope:.3f}：残差二乗和 {ssr:,.1f}')

推定値（中央の行）で残差二乗和が最も小さく、係数を大きくしても小さくしても増える。次に、残差と各説明変数の相関係数を計算し、散布図でも確かめる。

In [ ]:
display(pd.DataFrame({'残差との相関係数': [
    np.corrcoef(df['income'], multiple.resid)[0, 1],
    np.corrcoef(df['household_size'], multiple.resid)[0, 1],
]}, index=['所得', '世帯人数']))

fig, ax = plt.subplots()
ax.scatter(df['income'], multiple.resid, s=25, alpha=0.65)
ax.axhline(0, color='#bc3b45', linewidth=1)
ax.set(xlabel='Income (10,000 JPY / month)', ylabel='Residual (10,000 JPY / month)')
format_axes(ax)
ax.yaxis.set_major_locator(MultipleLocator(2))   # 残差の軸は2万円ごとに目盛を置く
plt.tight_layout()
plt.show()

相関係数はどちらも0で、散布図でも、所得が高くなるにつれて残差が上や下に偏る様子は見られない。

この性質は、仮定4の「誤差は所得や世帯人数と関係しない」をデータの中で再現したものと見ることができる。仮定4が成り立つなら、誤差と同じ性質を残差に持たせる OLS は自然な推定方法である。一方、OLS はどのようなデータでも残差をこの形にするので、次の例のように、仮定4が成り立たないときにもこの性質は成り立つ。

### 4.2. 残差から仮定4を確かめられない理由

世帯人数を省き、消費を所得だけで説明するモデルを考える。このモデルでは、世帯人数による消費の差が誤差に入る。

$$y_i=\beta_0+\beta_1x_{1i}+v_i,\qquad v_i=\beta_2x_{2i}+u_i$$

世帯人数は所得と相関するので、誤差 $v_i$ も所得と相関し、仮定4が成り立たない。一方、所得だけの回帰の残差も4.1節の性質を満たすので、所得との相関係数は必ず0になる。

In [ ]:
X_simple = sm.add_constant(df[['income']])
simple = sm.OLS(df['consumption'], X_simple).fit()

error_v = 2 * df['household_size'] + true_error   # 所得だけの式の誤差 v
display(pd.DataFrame({'所得との相関係数': [
    np.corrcoef(df['income'], error_v)[0, 1],
    np.corrcoef(df['income'], simple.resid)[0, 1],
]}, index=['所得だけの式の誤差 v', '所得だけの回帰の残差']))

誤差と所得の相関係数は約0.23、残差と所得の相関係数は0である。残差が説明変数と相関しないのは OLS の計算の結果なので、仮定4が成り立つ根拠にはならない。

OLS は、誤差 $v_i$ に含まれる世帯人数の分まで所得の係数に取り込んで、残差と所得の相関を0にする。そのため所得の係数が真の値からずれる。このずれは第6節で調べる。

**確認**：第2回の住宅データで残差を計算できても、誤差の条件付き平均が0と確認したことにはならないのはなぜでしょうか。観測されたものと仮定したものを分けて説明してください。

### 記入欄：誤差と残差

ここをダブルクリックし、考えたことを記入してください。`Shift + Enter` で表示に戻ります。

-

## 5. 係数のばらつきと説明変数どうしの相関

仮定1〜5の下で、重回帰の所得の係数の分散は次の式で表される。導き方は省略し、式が表す内容を読む。

$$\mathrm{Var}(\hat\beta_1)=\frac{\sigma^2}{SST_1\times(1-R_1^2)}$$

- $\sigma^2$：誤差の分散
- $SST_1=\sum_{i=1}^{n}(x_{1i}-\bar x_1)^2$：所得の偏差の二乗和。所得のばらつきが大きいほど、世帯数が多いほど大きい
- $R_1^2$：所得を世帯人数に回帰したときの決定係数。所得のばらつきのうち、世帯人数で説明できる割合である

分母の $SST_1\times(1-R_1^2)$ は、所得のばらつきのうち世帯人数では説明できない部分の大きさを表す。重回帰で世帯人数を一定にした比較に使えるのは、この部分だけである。

したがって、係数のばらつきが大きくなるのは次のときである。

- 誤差の分散 $\sigma^2$ が大きいとき
- 所得のばらつき $SST_1$ が小さいとき（世帯数が少ないときを含む）
- 所得が世帯人数でよく説明され、$R_1^2$ が1に近いとき

最後の場合が多重共線性にあたる。$R_1^2=1$ では分母が0になり、仮定3が成り立たない。

次のコードでは、所得を世帯人数に単回帰して $R_1^2$ を求める。

In [ ]:
X_aux = sm.add_constant(df[['household_size']])
aux = sm.OLS(df['income'], X_aux).fit()
r2_1 = aux.rsquared
print(f'所得を世帯人数に回帰した決定係数 R1^2：{r2_1:.3f}')
print(f'世帯人数では説明できない割合 1 - R1^2：{1 - r2_1:.3f}')

所得のばらつきのうち、世帯人数で説明できるのは約19%で、残りの約81%を比較に使える。このデータでは説明変数どうしの相関は強くない。

実際の分析では $\sigma^2$ が分からないので、残差の二乗和を $n-3$ で割った値で推定する（3は推定した係数の数）。statsmodels では `multiple.scale` がこの推定値である。式の $\sigma^2$ をこの推定値に置き換えて平方根をとると、statsmodels が表示する標準誤差になる。

In [ ]:
sst1 = ((df['income'] - df['income'].mean())**2).sum()
sigma_hat = np.sqrt(multiple.scale)
se_formula = sigma_hat / np.sqrt(sst1 * (1 - r2_1))
print(f'式から計算した標準誤差：{se_formula:.4f}')
print(f'statsmodels の標準誤差：{multiple.bse["income"]:.4f}')

二つの値は一致する。第2回の二つの人口密度は相関係数が約0.95と強く、一方の密度のばらつきの大部分がもう一方で説明できた。重回帰で標準誤差が大きくなったのは、この式の $1-R_1^2$ が小さかったためである。

相関が強いという理由で説明変数を削除すると、その変数の係数が0でない限り、次の節で見るように残した変数の係数がずれる。まず変数の定義・単位・重複を調べ、知りたい係数を明確にする。データを増やして $SST_1$ を大きくすることも考えられる。

## 6. 標本を取り直したときの推定値の分布

所得と世帯人数を固定し、誤差 $u_i$ だけを1,000回作り直す。標本ごとに、重回帰の所得の係数と、世帯人数を省いて消費を所得だけに回帰した傾きを計算する。

重回帰では仮定1〜4が成り立つので、推定値の平均は真の係数0.3になると考えられる。所得だけの回帰では、4.2節で見たように仮定4が成り立たない。このとき、所得だけの回帰の傾きの平均は次の値になる（導き方は省略する）。

> 真の所得の係数 ＋ 真の世帯人数の係数 × 世帯人数を所得に回帰した傾き

最後の傾きは、所得が1万円高い世帯で世帯人数が平均的に何人多いかを表す。

In [ ]:
size_on_income = sm.OLS(df['household_size'], X_simple).fit().params['income']
print(f'世帯人数を所得に回帰した傾き：{size_on_income:.4f}')
print(f'所得だけの回帰の傾きの平均（予想）：{0.3 + 2 * size_on_income:.3f}')

傾きは約0.024なので、所得だけの回帰の傾きの平均は $0.3+2\times0.024\approx0.35$ と予想できる。必要な説明変数を省いたことで生じるこの系統的なずれ（ここでは $2\times0.024\approx0.05$）を**欠落変数バイアス**という。所得の高い世帯ほど世帯人数が多く、所得が同じなら世帯人数が多いほど消費も多いため、所得の係数に世帯人数の分が加わる。

**実行前の予想**：重回帰と所得だけの回帰の所得の係数は、それぞれどの値の周りに分布するでしょうか。どちらの分布が広いでしょうか。

`np.linalg.pinv(X)` は、$y$ の値を並べたものに掛けると OLS の係数が得られる行列を返す。説明変数は固定しているので、この行列を一度だけ計算し、1,000回の推定に使う。

In [ ]:
rng_repeat = np.random.default_rng(202611)
B = 1000
pinv_multiple = np.linalg.pinv(X_multiple.to_numpy())
pinv_simple = np.linalg.pinv(X_simple.to_numpy())
mean_consumption = 5 + 0.3 * df['income'].to_numpy() + 2 * df['household_size'].to_numpy()

slopes_multiple, slopes_simple = [], []
for _ in range(B):
    y_repeat = mean_consumption + rng_repeat.normal(0, 4, n)
    slopes_multiple.append((pinv_multiple @ y_repeat)[1])
    slopes_simple.append((pinv_simple @ y_repeat)[1])

display(pd.DataFrame({
    '推定値の平均': [np.mean(slopes_multiple), np.mean(slopes_simple)],
    '予想した平均': [0.3, 0.3 + 2 * size_on_income],
    '推定値の標準偏差': [np.std(slopes_multiple, ddof=1), np.std(slopes_simple, ddof=1)],
}, index=['重回帰の所得の係数', '所得だけの回帰の傾き']))

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(min(slopes_multiple + slopes_simple), max(slopes_multiple + slopes_simple), 40)
ax.hist(slopes_multiple, bins=bins, alpha=0.6, label='Multiple regression')
ax.hist(slopes_simple, bins=bins, alpha=0.6, label='Simple regression (household size omitted)')
ax.axvline(0.3, color='#bc3b45', linestyle='--', label='True income coefficient')
ax.set(xlabel='Estimated income coefficient', ylabel='Number of samples')
format_axes(ax, x_money=False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

重回帰の推定値は0.3の周りに、所得だけの回帰の推定値は約0.35の周りに分布する。所得だけの回帰の分布は少し狭いが、中心は真の係数から離れている。分散の小ささだけでは、推定方法の良し悪しを判断できない。

1,000回の平均や標準偏差も乱数によって変動する。この実験は、仮定の下で成り立つ性質を数値で確かめたものである。

## 7. 演習：式と仮定から考える

1. 月額所得と、その12倍で作った年額所得を同時に回帰式に入れた。どの仮定が成り立たないか、どう直せばよいかを説明してください。
2. 第2回の住宅データでは、二つの対数人口密度の相関係数が約0.95で、総面積密度の係数の標準誤差は、単回帰の1.85から重回帰では6.12に増えた。第5節の式のどの部分が変わったためか、言葉で説明してください。
3. 所得だけの回帰の残差と所得の相関係数が0だったので、「誤差の条件付き平均が0という仮定を確認した」と報告した。この報告の問題点を説明してください。
4. 消費を所得だけに回帰した。省いた変数「資産」は、所得が同じなら消費を増やし、所得の高い世帯ほど多いとする。所得の係数は真の値より大きくなりやすいでしょうか、小さくなりやすいでしょうか。第6節の式を使って説明してください。

### 記入欄：演習

ここをダブルクリックし、考えたことを記入してください。`Shift + Enter` で表示に戻ります。

-

<details>
<summary>考えたあとに開く：ヒント・確認</summary>

1. 年額所得は月額所得の12倍なので、二つの係数を一つに決められない。仮定3が成り立たない例である。どちらか一方だけを使う。
2. 一方の密度のばらつきの大部分がもう一方の密度で説明できるので、$R_1^2$ が1に近く、分母の $1-R_1^2$ が小さくなった。そのため分散と標準誤差が大きくなった。
3. 残差と説明変数が相関しないことは、4.1節の性質から必ず成り立つ。仮定4は観測できない誤差についての仮定で、4.2節の例のように、仮定4が成り立たないモデルでも残差と説明変数の相関は0になる。
4. 資産の係数は正、資産を所得に回帰した傾きも正なので、その積は正になる。所得の係数は真の値より大きくなりやすい。

</details>

## 関数の資料

- [statsmodels：OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html)
- [NumPy：乱数の生成（Generator）](https://numpy.org/doc/stable/reference/random/generator.html)
- [NumPy：numpy.linalg.pinv](https://numpy.org/doc/stable/reference/generated/numpy.linalg.pinv.html)